# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pip install -q schedule pytest # установка библиотек, если ещё не

In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
import re
import pickle
import json

## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
def get_book_data(book_url: str) -> dict:
    """
    Получает данные о книге с одной страницы.

    Собирает информацию о книге: название, цену, рейтинг, количество в наличии,
    описание и дополнительные характеристики из таблицы Product Information.

    Args:
        book_url (str): адрес страницы для парсинга с информацией о книге

    Returns:
        dict: Словарь с данными о книге:
            - 'title' (str): название книги
            - 'price' (float): цена
            - 'rating' (str): рейтинг в звездах
            - 'availability' (str): информация о наличии
            - 'description' (str): описание книги
            - 'product_info' (dict): словарь с дополнительной информацией

    Raises:
        requests.RequestException: При ошибках сетевого запроса
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    response = requests.get(book_url)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    result = dict()

    # парсим название книги
    result['title'] = soup.find('h1').get_text()

    # переходим к таблице с описанием книги
    table = soup.find('table', class_="table table-striped")

    # нашли цену
    price_raw = (
        table.find("th", string="Price (incl. tax)")
        .find_parent("tr")
        .find("td")
        .get_text()
    )
    result['price'] = float(price_raw[1:])

    # рейтинг в звездах
    result['rating'] = (
        soup.find("article", class_="product_page")
        .find("div", class_="col-sm-6 product_main")
        .find("p", class_=re.compile(r"star-rating.*"))
        .get('class')[1]
    )

    # информация о наличии
    result['availability'] = (
        table.find("th", string="Availability")
        .find_parent("tr")
        .find("td")
        .get_text()
    )

    # описание книги

    try:
        result['description'] = (
            soup.find("article", class_="product_page")
            .find("div", id="product_description")
            .find_next_sibling("p")
            .get_text()
        )
    except AttributeError:
        result['description'] = ""

    # словарь с дополнительной информацией
    ks = [i.get_text() for i in table.find_all("th")]
    vs = [i.get_text() for i in table.find_all("td")]
    result['product_info'] = dict(zip(ks, vs))

    return result
# КОНЕЦ ВАШЕГО РЕШЕНИЯ


In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': 51.77,
 'rating': 'Three',
 'availability': 'In stock (22 available)',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe pla

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def scrape_catalog(debug: bool = False) -> list:
    """
    Функция парсит каталог и возвращает список ссылок на отдельные книги.

    Args
        debug (bool): при отладке читает данные из файла, по умолчанию False

    Returns
        book_urls (list): список URL адресов страниц с книгами
    """
    start_time = time.time()
    n = 1
    pages = []

    # Для удобства отладки добавим опцию писать/читать каталог в файл/из файла
    # Вне режима дебага парсим каталог каждый раз
    if not debug:
        while True:
            try:
                response = requests.get(
                    f"http://books.toscrape.com/catalogue/page-{n}.html",
                    timeout=10
                )

                if not response.ok:
                    break

                pages.append(response.content)
                #print(f"Catalog page {n} scraped")
                n += 1

            except requests.exceptions.RequestException:
                print("Connection error, retrying...")
                time.sleep(5)
                continue

        # with open('catalog_pages.pkl', 'wb') as f:
        #     pickle.dump(pages, f)

    elif debug:

        with open('catalog_pages.pkl', 'rb') as f:
            pages = pickle.load(f)

        print(f"Loaded {len(pages)} pages from file")

    base_url = 'http://books.toscrape.com/catalogue/'
    book_urls = []
    for page in pages:
        soup = BeautifulSoup(page, 'html.parser')

        table = (
            soup.find("div", class_="col-sm-8 col-md-9")
            .find_all("li", class_="col-xs-6 col-sm-4 col-md-3 col-lg-3")
        )

        for book in table:
            # book.find("a", href=True).get("href")
            book_urls.append(base_url + book.find("a", href=True).get("href"))

    end_time = time.time()

    execution_time = end_time - start_time
    print(f"Catalog scraping took {execution_time:.2f} seconds")

    return book_urls


def scrape_books(book_urls: list, is_save: bool = False,
                 debug: bool = False) -> list:
    """
    Парсит предоставленные URL со страницами книг, достает описание этих книг.
    Может записать полученную информацию в файл.

    Args:
        book_urls (list): список URL-адресов книг из каталога
        is_save (bool): сохранение инфо о книгах в файл, по умолчанию false
        debug (bool): режим отладки, парсит первые 10 книг, по умолчанию false
    Return:
        books_data (list): список словарей из scrape_catalog() с книгами
    """

    # Для отладки парсим первые 10 книг для экономии времени
    if debug:
        book_urls = book_urls[0:10]

    books_data = []
    start_time = time.time()

    for i, book_url in enumerate(book_urls, 1):
        for attempt in range(3):  # Ретрай до 3х раз
            try:
                book = get_book_data(book_url)
                books_data.append(book)
                break  # Успех, переходим к следующей книге
            except requests.exceptions.RequestException:
                if attempt < 2:  # Не последняя попытка
                    print(
                        f"Failed to load {book_url}, retry attempt {attempt}")
                    time.sleep(2)
                    continue
                print(f"Failed to load {book_url} after 3 attempts")

        #print(f"Loaded book No {i}: {book_url}")

    # Пишем данные в файл
    # Отладочные данные не надо писать в файл
    if is_save and not debug:
        with open('artifacts/books_data.txt', 'w', encoding='utf-8') as f:
            json.dump(books_data, f, indent=4, ensure_ascii=False)
            print("Scraped books saved to books_data.txt")

    end_time = time.time()

    execution_time = end_time - start_time
    print(f"Catalog scraping took {execution_time:.2f} seconds")

    return books_data
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [21]:
# Проверка работоспособности функции
books = scrape_catalog(debug=False)
res = scrape_books(is_save=True, book_urls=books, debug=False) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Catalog scraping took 27.49 seconds


FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/books_data.txt'

## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

# КОНЕЦ ВАШЕГО РЕШЕНИЯ

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [ ]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest test/test_scraper.py

## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```